# Notebook 20 — Temporal robustness and paired model diagnostics

**Development-only experiment.** Notebook 19's two-fold smoke test showed calibrated stacking with mean ROC-AUC ≈ 0.5714, compared with Random Forest ≈ 0.5667. That difference is preliminary. This notebook can rerun the original *nested chronological* pipeline on five folds and then produce paired fold diagnostics. It does **not** use the historical final holdout or prospective ledger.

**Important:** Expanding-window folds share training observations. Fold-level differences are descriptive, not independent observations suitable for conventional significance tests.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src').exists(): raise FileNotFoundError('Run from repository root or notebooks/')
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.temporal_calibration import run_temporal_calibration
from stock_ml_dynamic.temporal_stability import write_diagnostics, paired_fold_deltas
from stock_ml_dynamic.pooled_models import load_master
print('Project root:',ROOT)

## 1. Configuration

The default loads the existing Notebook 19 fold metrics for a quick diagnostic check. Set `RUN_FULL_BENCHMARK=True` to execute the **five-fold** benchmark with its existing inner temporal stacking and calibration procedure. This is compute-intensive; run on your local machine. No model selection is made using the final holdout.

In [ ]:
RUN_FULL_BENCHMARK = False
N_FOLDS = 5
MAX_TRAIN_ROWS = 60000
TREE_TRAIN_ROWS = 35000
SEED = 42
SOURCE_DIR = ROOT/'artifacts'/'temporal_calibration'/'1d'
OUTPUT_DIR = ROOT/'artifacts'/'temporal_stability'/'1d'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Full benchmark enabled:',RUN_FULL_BENCHMARK)

In [ ]:
if RUN_FULL_BENCHMARK:
    required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
    candidates=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
    master_dir=next((p for p in candidates if all((p/f).exists() for f in required)),None)
    if master_dir is None: raise FileNotFoundError('Notebook 03 master artifacts are required')
    master,latest,bundle,source_manifest=load_master(master_dir)
    scores,summary,fold_manifest,boundary=run_temporal_calibration(
        master,n_folds=N_FOLDS,max_train_rows=MAX_TRAIN_ROWS,
        tree_train_rows=TREE_TRAIN_ROWS,seed=SEED,output_dir=OUTPUT_DIR/'benchmark')
    print('Development ends:',boundary)
else:
    source=SOURCE_DIR/'fold_metrics.csv'
    if not source.exists():
        raise FileNotFoundError(f'{source} not found. Run Notebook 19 first, or enable RUN_FULL_BENCHMARK.')
    scores=pd.read_csv(source)
    print('Loaded existing Notebook 19 scores:',source)
    print('Observed folds:',scores.fold.nunique(),'(two folds are smoke-test evidence only)')
print('Rows:',len(scores),'models:',scores.model.nunique())

## 2. Paired fold comparisons against Random Forest

All improvements are oriented so that **positive is better**. For log-loss and ECE, lower is better, so their differences are sign-reversed. This does not imply statistical significance.

In [ ]:
paired,winners,diagnostic_manifest=write_diagnostics(scores,OUTPUT_DIR,reference='random_forest')
show=['candidate','n_folds','mean_improvement_roc_auc','min_improvement_roc_auc','positive_folds_roc_auc','mean_improvement_log_loss','positive_folds_log_loss']
display(paired[show])
display(winners)
print('Diagnostics saved to:',OUTPUT_DIR)

## 3. Model performance over chronological folds

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
for model,frame in scores.groupby('model'):
    frame=frame.sort_values('fold')
    ax.plot(frame.fold,frame.roc_auc,marker='o',label=model)
ax.set_xlabel('Chronological outer fold')
ax.set_ylabel('Macro one-vs-rest ROC-AUC')
ax.set_title('Development-fold discrimination (not final holdout)')
ax.legend(fontsize=8,ncol=2)
ax.grid(alpha=.2)
plt.tight_layout();plt.show()

## 4. Paired ROC-AUC improvement by fold

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
for candidate in ['temporal_stack_calibrated','soft_vote_calibrated','logistic_l2']:
    if candidate in set(scores.model):
        frame=paired_fold_deltas(scores,candidate,'random_forest')
        ax.plot(frame.fold,frame.improvement_roc_auc,marker='o',label=candidate)
ax.axhline(0,linestyle='--',linewidth=1)
ax.set_xlabel('Chronological outer fold')
ax.set_ylabel('Paired ROC-AUC improvement over Random Forest')
ax.legend(fontsize=8)
ax.grid(alpha=.2)
plt.tight_layout();plt.show()

## 5. Probability quality and decision checkpoint

Review **ROC-AUC, log-loss, ECE, and fold consistency together**. A model can rank outcomes better yet choose the correct top class less often. Temperature scaling does not change the argmax class. Do not use these diagnostics to reopen Notebook 13's holdout.

**Decision rule for follow-up:** if the five-fold calibrated stack consistently improves ROC-AUC *and* log-loss, prioritize prospective shadow evaluation alongside the frozen Logistic L2 model, with separate model versioning. If the improvements are unstable, favor the simpler RF or soft-vote benchmark. This is a research prioritization rule, not a trading signal.

In [ ]:
quality=scores.groupby('model',as_index=False).agg(
    folds=('fold','nunique'),roc_auc_mean=('roc_auc','mean'),
    accuracy_mean=('accuracy','mean'),balanced_accuracy_mean=('balanced_accuracy','mean'),
    log_loss_mean=('log_loss','mean'),ece_mean=('ece','mean')).sort_values('roc_auc_mean',ascending=False)
display(quality)
quality.to_csv(OUTPUT_DIR/'quality_summary.csv',index=False)
print('Historical final holdout: NOT ACCESSED')
print('Prospective ledger: NOT ACCESSED')